<a href="https://colab.research.google.com/github/riraonline/data_science_2026/blob/main/Pertemuan12_Richardin_Rafsanjani_220401010116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Generate & Eksplorasi Dataset Transaksi**

In [3]:
import warnings
# Mengabaikan semua warning agar output terlihat bersih
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning) # Opsional, untuk library mlxtend/sklearn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity

# Set seed agar hasil random konsisten
np.random.seed(42)

produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk secara acak
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola tersembunyi: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh 3 transaksi pertama:', transaksi[:3])
print('Jumlah total transaksi:', len(transaksi))
print("-" * 50)

Contoh 3 transaksi pertama: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah total transaksi: 50
--------------------------------------------------


**One-Hot Encoding Transaksi**

In [4]:
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("5 Baris Pertama Dataframe One-Hot Encoding:")
print(df.head())

5 Baris Pertama Dataframe One-Hot Encoding:
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


**Cari Frequent Itemset dengan Apriori**

In [5]:
# Eksperimen dengan beberapa nilai min_support
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support = 0.1 untuk analisis selanjutnya
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print("\nTop 10 Frequent Itemset:")
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemset:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


**Bentuk & Saring Aturan Asosiasi**

In [6]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("Top 10 Association Rules yang Valid (Lift > 1):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Top 10 Association Rules yang Valid (Lift > 1):
         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


**Rekomender Sederhana dengan Content-Based Filtering**

In [7]:
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy', 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

# Dapatkan fitur kategori dengan one-hot encoding
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    # Hapus produk itu sendiri dari daftar rekomendasi
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print("Rekomendasi Content-Based yang mirip dengan 'Roti':", rekomendasi_serupa('Roti'))

Rekomendasi Content-Based yang mirip dengan 'Roti': ['Selai', 'Sereal', 'Susu']


**Bandingkan Kedua Pendekatan**

In [8]:
produk_target = 'Roti'

# Cari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print(f"Rekomendasi dari Association Rules untuk '{produk_target}':")
print(rules_terkait[['consequents', 'lift']].head())

print(f"\nRekomendasi dari Content-Based untuk '{produk_target}':")
print(rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules untuk 'Roti':
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based untuk 'Roti':
['Selai', 'Sereal', 'Susu']
